In [ ]:
import os
import re
import ast
import json
import glob
import time
import random
from typing import Optional, Dict, Any, List, Tuple

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import openai
from openai import AsyncOpenAI
from openai import RateLimitError, APITimeoutError, APIConnectionError, InternalServerError
from pydantic import BaseModel, Field, confloat

print("openai version:", getattr(openai, "__version__", "unknown"))


In [ ]:
api_key = ''

In [ ]:

# -----------------------------
# Paths / discovery
# -----------------------------
OUTPUTS_ROOT = "../outputs"
QUESTION_MODEL_DIR = "Qwen3-0.6B"   # fix this for loading questions, as you requested

# This matches your example filename; adjust if you want other n/temp/k variants.
CSV_GLOB = "n_1000_temp_0.7_k_100_out_df_temp_0.6_k_1.csv"

# Where to write baselines
BASELINE_ROOT = os.path.join(OUTPUTS_ROOT, "_baselines_openai")

# -----------------------------
# OpenAI baseline model
# -----------------------------
OPENAI_MODEL = "gpt-4o-mini"
TEMPERATURE = 0.0

# Concurrency: main speed knob (start 10–20; tune if rate-limited)
CONCURRENCY = 20

# checkpoint every N completed questions
SAVE_EVERY = 2000

# optionally limit for a pilot
MAX_Q_PER_DATASET: Optional[int] = None  # e.g. 50

aclient = AsyncOpenAI(timeout=60, max_retries=0, api_key=api_key)


In [ ]:
class BaselineOutput(BaseModel):
    answer: str = Field(..., description="Short final answer (e.g., entity name or final numeric answer).")
    confidence: confloat(ge=0.0, le=1.0) = Field(..., description="P(answer is correct) in [0,1].")

SYSTEM_PROMPT_BASELINE = """You are answering questions.

Return:
- answer: a short final answer (no explanation; just the answer).
- confidence: a number in [0,1] representing the probability that your answer is correct.

Be honest and well-calibrated. If you are unsure, lower the confidence.
"""

def build_baseline_user_prompt(question: str) -> str:
    return f"Question:\n{question}\n\nReturn JSON with keys answer and confidence."

TRANSIENT = (RateLimitError, APITimeoutError, APIConnectionError, InternalServerError)

async def call_baseline_async(question: str, max_attempts: int = 8) -> BaselineOutput:
    inp = [
        {"role": "system", "content": SYSTEM_PROMPT_BASELINE},
        {"role": "user", "content": build_baseline_user_prompt(question)},
    ]

    for attempt in range(max_attempts):
        try:
            if "gpt4" in OPENAI_MODEL:
                resp = await aclient.responses.parse(
                    model=OPENAI_MODEL,
                    input=inp,
                    temperature=TEMPERATURE,
                    text_format=BaselineOutput,
                )
            else:
                resp = await aclient.responses.parse(
                    model=OPENAI_MODEL,
                    input=inp,
                    text_format=BaselineOutput,
                )
            return resp.output_parsed
        except TRANSIENT:
            # exponential backoff + jitter
            base = 0.5 * (2 ** attempt)
            sleep = min(20.0, base) + random.random() * 0.2
            await asyncio.sleep(sleep)
        except Exception:
            # permanent error: surface immediately
            raise

    raise RuntimeError(f"Baseline call failed after {max_attempts} attempts.")


In [ ]:
def discover_dataset_csvs() -> Dict[str, str]:
    """
    Returns {dataset_name: csv_path} for all datasets under outputs/Qwen3-0.6B/*/<CSV_GLOB>.
    """
    pattern = os.path.join(OUTPUTS_ROOT, QUESTION_MODEL_DIR, "*", CSV_GLOB)
    paths = sorted(glob.glob(pattern))
    if not paths:
        raise FileNotFoundError(f"No files matched: {pattern}")

    out = {}
    for p in paths:
        # .../outputs/Qwen3-0.6B/<dataset>/<file>.csv
        dataset = os.path.basename(os.path.dirname(p))
        out[dataset] = p
    return out

def load_questions_df(csv_path: str, max_q: Optional[int] = None) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    needed = {"id", "question", "ground_truth"}
    missing = needed - set(df.columns)
    if missing:
        raise ValueError(f"{csv_path} missing columns: {missing}")

    keep = df[["id", "question", "ground_truth"]].copy()
    if max_q is not None:
        keep = keep.iloc[:max_q].copy()
    keep["id"] = keep["id"].astype(str)
    keep["question"] = keep["question"].astype(str)
    return keep


In [ ]:
_NUM_RE = re.compile(r"-?\d+(?:\.\d+)?")

def _strip_latex(s: str) -> str:
    s = str(s)
    # pull out \boxed{...}
    s = re.sub(r"\\boxed\{([^}]*)\}", r"\1", s)
    # pull out \text{...}
    s = re.sub(r"\\text\{([^}]*)\}", r"\1", s)
    # remove latex slashes / dollars
    s = s.replace("\\", " ").replace("$", " ")
    return s

def norm_str(s: str) -> str:
    s = _strip_latex(s)
    s = s.strip().lower()
    # normalize whitespace
    s = re.sub(r"\s+", " ", s)
    # trim common punctuation at ends
    s = s.strip(" \n\t\r.,;:!?'\"()[]{}")
    return s

def extract_last_number(s: str) -> Optional[str]:
    s = _strip_latex(s)
    s = s.replace(",", "")
    nums = _NUM_RE.findall(s)
    if not nums:
        return None
    return nums[-1]

def parse_aliases(gt) -> List[str]:
    """
    Trivia/WebQ ground_truth often is list-like; try literal_eval if it's a string that looks like a list.
    """
    if isinstance(gt, list):
        return [str(x) for x in gt]
    if isinstance(gt, str):
        t = gt.strip()
        if (t.startswith("[") and t.endswith("]")) or (t.startswith("(") and t.endswith(")")):
            try:
                v = ast.literal_eval(t)
                if isinstance(v, (list, tuple)):
                    return [str(x) for x in v]
            except Exception:
                pass
        # fall back: single alias
        return [gt]
    return [str(gt)]

def trivia_correct(pred: str, aliases: List[str]) -> int:
    if pred is None:
        return 0
    a = norm_str(pred)
    for gt in aliases:
        t = norm_str(gt)
        if (a in t) or (t in a):
            return 1
    return 0

def baseline_correct(dataset: str, pred: str, gt) -> int:
    """
    Dataset-aware correctness.
    - trivia_qa/webq: substring match over aliases
    - sciq: substring match over single ground truth string
    - gsm8k/math: compare extracted final number
    - else: normalized exact match
    """
    if dataset in {"trivia_qa", "webq"}:
        return trivia_correct(pred, parse_aliases(gt))

    if dataset == "sciq":
        return trivia_correct(pred, [str(gt)])

    if dataset in {"gsm8k", "math"}:
        pnum = extract_last_number(pred)
        gnum = extract_last_number(gt)
        if pnum is None or gnum is None:
            # fall back to normalized exact match
            return int(norm_str(pred) == norm_str(gt))
        return int(pnum == gnum)

    return int(norm_str(pred) == norm_str(gt))


In [ ]:
import asyncio

async def run_baseline_for_dataset(
    dataset: str,
    qdf: pd.DataFrame,
    out_dir: str,
    concurrency: int = CONCURRENCY,
    save_every: int = SAVE_EVERY,
) -> pd.DataFrame:
    os.makedirs(out_dir, exist_ok=True)
    out_csv = os.path.join(out_dir, f"baseline_{OPENAI_MODEL}.csv")
    out_parquet = os.path.join(out_dir, f"baseline_{OPENAI_MODEL}.parquet")

    # resumable: load existing
    done = set()
    rows: List[Dict[str, Any]] = []
    if os.path.exists(out_parquet):
        prev = pd.read_parquet(out_parquet)
        rows = prev.to_dict("records")
        done = set(prev["id"].astype(str).tolist())
        print(f"[{dataset}] resuming from {len(done)} completed questions")

    jobs = []
    for _, r in qdf.iterrows():
        rid = str(r["id"])
        if rid in done:
            continue
        jobs.append((rid, str(r["question"]), r["ground_truth"]))

    print(f"[{dataset}] queued {len(jobs)} baseline calls (concurrency={concurrency})")

    sem = asyncio.Semaphore(concurrency)
    pbar = tqdm(total=len(jobs), desc=f"{dataset} baselines")

    async def one(job):
        rid, question, gt = job
        async with sem:
            out = await call_baseline_async(question)
        corr = baseline_correct(dataset, out.answer, gt)
        return {
            "id": rid,
            "dataset": dataset,
            "question": question,
            "ground_truth": gt,
            "model": OPENAI_MODEL,
            "answer": out.answer,
            "confidence": float(out.confidence),
            "correct": int(corr),
        }

    n_since_save = 0
    tasks = [asyncio.create_task(one(j)) for j in jobs]

    for fut in asyncio.as_completed(tasks):
        rec = await fut
        rows.append(rec)
        done.add(rec["id"])
        n_since_save += 1
        pbar.update(1)

        if n_since_save >= save_every:
            df_out = pd.DataFrame(rows)
            df_out.to_parquet(out_parquet, index=False)
            df_out.to_csv(out_csv, index=False)
            n_since_save = 0

    pbar.close()

    df_out = pd.DataFrame(rows).sort_values(by='id')
    df_out.to_parquet(out_parquet, index=False)
    df_out.to_csv(out_csv, index=False)
    return df_out


In [ ]:
dataset_to_csv = discover_dataset_csvs()
print("Found datasets:", sorted(dataset_to_csv.keys()))

exp_datasets = [
    "gsm8k",
    "polymath",
    "trivia_qa",
    "sciq",
    "webq"
]
to_delete = []
for k, v in dataset_to_csv.items():
    if k not in exp_datasets:
        to_delete.append(k)
for k in to_delete:
    del dataset_to_csv[k]

print("Running datasets:", sorted(dataset_to_csv.keys()))

# Run all datasets sequentially (each dataset uses internal async concurrency)
all_outputs = {}
for dataset, csv_path in dataset_to_csv.items():
    qdf = load_questions_df(csv_path, max_q=MAX_Q_PER_DATASET)
    out_dir = os.path.join(BASELINE_ROOT, OPENAI_MODEL, dataset)
    df_out = await run_baseline_for_dataset(dataset, qdf, out_dir)
    all_outputs[dataset] = df_out

# Optional: combined view
baseline_all = pd.concat(all_outputs.values(), ignore_index=True)
baseline_all.head()
